# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using the `mlcroissant` library. All schema entities (record sets, fields, columns) are referenced by their Croissant schema `@id` fields for clarity and traceability.

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore the dataset structure using `mlcroissant`. The metadata contains dataset information and schema definitions.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Get metadata object
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
List available record sets and their fields using their Croissant `@id`. We'll also show a summary for each record set's structure.

In [ ]:
# List all record sets: use metadata.record_sets
print('Available Record Set @id values:')
for rs in metadata.record_sets:
    print(f"- {rs.id}")

print('\nListing fields for each record set:')
for rs in metadata.record_sets:
    print(f"\nRecordSet @id: {rs.id}\n  Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name} | type: {field.data_type}")

## 3. Data Extraction
Let's load the main survey data table into a pandas DataFrame. We'll identify the record set by its `@id` and extract its records for further analysis.

**Note:** Use the `@id` of the main record set as discovered above. For this dataset, the main survey data is in record set `cr:recordSet/survey_data` (example, update according to listing above).

In [ ]:
# List all record set @ids
record_sets = [rs.id for rs in metadata.record_sets]
print('Record Set @ids to extract:', record_sets)

# Dictionary to hold all dataframes
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  -> {len(df)} records loaded. Columns:", df.columns.tolist())

# For exploration, choose the main survey record set. Change this if necessary based on overview above.
main_record_set_id = record_sets[0]  # use the first one by default

print(f"\nFirst rows of main record set '{main_record_set_id}':")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, we'll perform basic EDA using one of the numeric fields (e.g., PHQ-9 or GAD-7 score field by its `@id`).

**Steps:**
- Filter rows with high scores (e.g., PHQ-9 total > 10)
- Normalize the numeric field
- Group by an attribute (e.g., gender or education level) if present

Edit variable assignments below to match appropriate field `@id`s discovered above.

In [ ]:
# Choose a numeric field for filtering & normalization. Update field @id as needed.
df = dataframes[main_record_set_id]
# Example: suppose field is 'cr:field/phq9_total' and 'cr:field/gender'
numeric_field_id = None
group_field_id = None

# Try to auto-detect likely numeric fields and a group field
for field in metadata.record_sets[0].fields:
    if field.data_type in ('Integer', 'Float', 'Number') and not numeric_field_id:
        numeric_field_id = field.id
    if (field.data_type == 'Text' or field.data_type == 'String') and not group_field_id:
        group_field_id = field.id

if not (numeric_field_id and group_field_id):
    print("Could not detect numeric or group field automatically. Please check field @id list above.")

print(f"Numeric field: {numeric_field_id}")
print(f"Group field: {group_field_id}")

# Remove rows with missing numeric values
df_clean = df.dropna(subset=[numeric_field_id])

threshold = 10
filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
print(f"Filtered: {len(filtered_df)} records with {numeric_field_id} > {threshold}\n")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized field:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the chosen field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and the grouping field using standard plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df_clean[numeric_field_id], kde=True, bins=20)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by group field if available
if group_field_id in df_clean.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df_clean[group_field_id], y=df_clean[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, you learned how to:
- Load a dataset described by a Croissant schema using `mlcroissant`
- Explore record sets and fields by their Croissant `@id`
- Extract, analyze, and visualize key variables from the dataset

You are encouraged to extend this notebook with custom analyses or to incorporate additional record sets and metadata fields for deeper research.